# Pair 13, RankGPT: reproducing Table 3 (NovelEval) with an open 7B model

Sun et al. (EMNLP 2023) show that a chat LLM can re-rank passages by reading 20 numbered passages and writing a
permutation (`[2] > [3] > [1]`). **NovelEval** (their Table 3) is a set of 21 questions collected in June 2023, after
the training data of the paper's GPT models, each with 20 judged passages. RankZephyr's base model (Mistral-7B,
September 2023) is newer, so for RankZephyr we cannot rule out exposure.

We replace GPT-4 (no API budget) by **RankZephyr-7B** (Pradeep et al. 2023, a Zephyr-7B fine-tuned on RankGPT
permutations) in 4-bit on an 8 GB laptop GPU. Every LLM window is cached in `reproduction/cache/listwise/`, so this
notebook replays the cached windows and needs no GPU (only RankZephyr's tokenizer, a few MB, is downloaded); delete
`reproduction/cache/listwise/rankzephyr_noveleval.jsonl` to regenerate them (about 25 minutes on an RTX 4060).

Requirements: the packages of `requirements.txt` in this folder; run the notebook from this folder. `reproduction/`
holds the code, the NovelEval data (the 21 questions, from the RankGPT repository) and the cached windows;
`RANKGPT_REPRO` can point at our full research repository instead.

In [1]:
import json, os, subprocess, sys
from pathlib import Path
from IPython.display import Markdown, display

HERE = Path.cwd().resolve()   # the notebook runs in pairs/13-rankgpt
REPRO = Path(os.environ['RANKGPT_REPRO']) if os.environ.get('RANKGPT_REPRO') else HERE / 'reproduction'
assert (REPRO / 'scripts' / '04_run_noveleval.py').exists(), f'run the notebook from pairs/13-rankgpt: {REPRO.name}/scripts/04_run_noveleval.py not found'
PY = sys.executable
env = dict(os.environ, PYTHONUTF8='1', PYTHONIOENCODING='utf-8')

def run(*args):
    r = subprocess.run([PY, '-P', *args], cwd=REPRO, env=env, capture_output=True, text=True, encoding='utf-8')
    print('\n'.join(l for l in (r.stdout + r.stderr).splitlines() if l.startswith('[') or 'nDCG' in l or 'Error' in l or 'Traceback' in l))
    return r.returncode

def pct(x):
    return f'{100*x:.2f}'
print('repo:', REPRO.name)

repo: reproduction


## 1. NovelEval: RankZephyr vs the paper's numbers

Three shuffled input orders (seeds 0–2, as the paper shuffles the 20 passages), plus the BM25 order and the order in
which the passages were released. nDCG@1/5/10 over the 21 questions.

In [2]:
rc = run('scripts/04_run_noveleval.py', '--tag', 'rankzephyr', '--seeds', '0', '1', '2', '--orders', 'shuffled', 'bm25', 'released')
assert rc == 0
r = json.load(open(REPRO / 'results' / 'noveleval_rankzephyr.json', encoding='utf8'))
paper = {'BM25 (paper)': (33.33, 45.96, 55.77), 'monoT5-3B (paper)': (83.33, 78.38, 84.62),
         'gpt-3.5-turbo (paper)': (76.19, 74.15, 75.71), 'gpt-4 (paper)': (85.71, 87.49, 90.45),
         'RankZephyr bf16 (RankZephyr paper)': (92.86, 86.15, 89.34)}
paper_order = {'gpt-3.5-turbo (paper)': 'shuffled', 'gpt-4 (paper)': 'shuffled', 'RankZephyr bf16 (RankZephyr paper)': 'released'}
rows = ['| system | input order | nDCG@1 | nDCG@5 | nDCG@10 |', '|---|---|---|---|---|']
for k, (a, b, d) in paper.items():
    rows.append(f'| {k} | {paper_order.get(k, "")} | {a:.2f} | {b:.2f} | {d:.2f} |')
for order, o in r['orders'].items():
    for s in o['runs']:
        seed = f" seed {s['seed']}" if s.get('seed') is not None else ''
        rows.append(f"| RankZephyr-7B 4-bit (ours) | {order}{seed} | {pct(s['nDCG@1'])} | {pct(s['nDCG@5'])} | {pct(s['nDCG@10'])} |")
    if len(o['runs']) > 1:
        a = o['aggregate']
        rows.append(f"| **RankZephyr-7B 4-bit (ours)** | **{order} mean ± std** | {pct(a['nDCG@1']['mean'])} ± {pct(a['nDCG@1']['std'])} | "
                    f"{pct(a['nDCG@5']['mean'])} ± {pct(a['nDCG@5']['std'])} | {pct(a['nDCG@10']['mean'])} ± {pct(a['nDCG@10']['std'])} |")
display(Markdown('\n'.join(rows)))

[noveleval] rankzephyr_shuffled_seed0: nDCG@1/5/10 = 88.10/89.18/89.97 well-formed 100% missing-ids 1 dup 0
[noveleval] rankzephyr_shuffled_seed1: nDCG@1/5/10 = 90.48/87.15/89.42 well-formed 100% missing-ids 1 dup 0
[noveleval] rankzephyr_shuffled_seed2: nDCG@1/5/10 = 83.33/84.93/88.12 well-formed 100% missing-ids 0 dup 0
[noveleval] shuffled: mean nDCG@1/5/10 = 87.30/87.09/89.17
[noveleval] rankzephyr_bm25: nDCG@1/5/10 = 83.33/85.22/88.89 well-formed 95% missing-ids 0 dup 0
[noveleval] bm25: mean nDCG@1/5/10 = 83.33/85.22/88.89
[noveleval] rankzephyr_released: nDCG@1/5/10 = 92.86/88.25/89.46 well-formed 100% missing-ids 0 dup 0
[noveleval] released: mean nDCG@1/5/10 = 92.86/88.25/89.46


| system | input order | nDCG@1 | nDCG@5 | nDCG@10 |
|---|---|---|---|---|
| BM25 (paper) |  | 33.33 | 45.96 | 55.77 |
| monoT5-3B (paper) |  | 83.33 | 78.38 | 84.62 |
| gpt-3.5-turbo (paper) | shuffled | 76.19 | 74.15 | 75.71 |
| gpt-4 (paper) | shuffled | 85.71 | 87.49 | 90.45 |
| RankZephyr bf16 (RankZephyr paper) | released | 92.86 | 86.15 | 89.34 |
| RankZephyr-7B 4-bit (ours) | shuffled seed 0 | 88.10 | 89.18 | 89.97 |
| RankZephyr-7B 4-bit (ours) | shuffled seed 1 | 90.48 | 87.15 | 89.42 |
| RankZephyr-7B 4-bit (ours) | shuffled seed 2 | 83.33 | 84.93 | 88.12 |
| **RankZephyr-7B 4-bit (ours)** | **shuffled mean ± std** | 87.30 ± 2.97 | 87.09 ± 1.74 | 89.17 ± 0.78 |
| RankZephyr-7B 4-bit (ours) | bm25 | 83.33 | 85.22 | 88.89 |
| RankZephyr-7B 4-bit (ours) | released | 92.86 | 88.25 | 89.46 |

Reading: the open 7B model comes within 1.3 points of the paper's GPT-4 (nDCG@10 89.17 vs 90.45 over three shuffled
orders; 21 questions), far above gpt-3.5-turbo (75.71). The input order moves nDCG@1 by up to 9.5 points (83.33 to
92.86) and nDCG@10 by up to 1.9 (88.12 to 89.97). 104 of the 105 windows were well-formed (one window of the BM25
order was not), and 2 of the 2,100 ids were missing (seeds 0 and 1).

## 2. Permutation distillation (Table 7, small scale) on the testbed

The paper distils ChatGPT's permutations of 10K MS MARCO queries into a cross-encoder with RankNet and shows that this
beats training the same model on the MS MARCO labels of the same queries. We did the same with a 22M MiniLM-L6 student
(`scripts/05_distill.py` of our research repository; the students, 175 MB, are not in this folder) and score both
students with the shared `testbed.py score --against ce`
(BM25 top-50 candidates, all queries). `ce` is the testbed's MiniLM cross-encoder trained on all of MS MARCO.

In [3]:
rows = ['| student (MiniLM-L6, 22M) | dataset | nDCG@10 | ce nDCG@10 | Δ vs ce | 95% CI | p |', '|---|---|---|---|---|---|---|']
for tag in ['minilm_chatgpt', 'minilm_marco_binary']:
    p = REPRO / 'results' / f'testbed_{tag}_top50.json'
    if not p.exists():
        print('missing', p.name, '- run scripts/08_testbed_score.py --tag', tag); continue
    t = json.load(open(p, encoding='utf8'))
    for ds, d in t['datasets'].items():
        v = d['ndcg10_vs_baseline']
        rows.append(f"| {t['method']} | {ds} | {d['ndcg10']:.3f} | {d['baseline']['ndcg10']:.3f} | {v['delta']:+.3f} | [{v['ci95'][0]:+.3f}, {v['ci95'][1]:+.3f}] | {v['p']:.2g} |")
display(Markdown('\n'.join(rows)))

| student (MiniLM-L6, 22M) | dataset | nDCG@10 | ce nDCG@10 | Δ vs ce | 95% CI | p |
|---|---|---|---|---|---|---|
| MiniLM-L6 distilled from ChatGPT permutations (10K q), BM25 top-50 | scifact | 0.645 | 0.688 | -0.043 | [-0.071, -0.016] | 0.0022 |
| MiniLM-L6 distilled from ChatGPT permutations (10K q), BM25 top-50 | nfcorpus | 0.336 | 0.347 | -0.011 | [-0.021, -0.002] | 0.024 |
| MiniLM-L6 distilled from ChatGPT permutations (10K q), BM25 top-50 | arguana | 0.279 | 0.422 | -0.144 | [-0.163, -0.125] | 7.4e-48 |
| MiniLM-L6 trained on MS MARCO labels (10K q), BM25 top-50 | scifact | 0.216 | 0.688 | -0.472 | [-0.517, -0.427] | 6.5e-60 |
| MiniLM-L6 trained on MS MARCO labels (10K q), BM25 top-50 | nfcorpus | 0.224 | 0.347 | -0.123 | [-0.143, -0.103] | 4.8e-27 |
| MiniLM-L6 trained on MS MARCO labels (10K q), BM25 top-50 | arguana | 0.025 | 0.422 | -0.397 | [-0.417, -0.377] | 7.9e-227 |

Reading: with the same architecture and the same 10K queries, ChatGPT's permutations give a usable re-ranker
(within 1–4 points of `ce` on SciFact / NFCorpus) while our student trained on the binary labels collapses. The
paper's Table 13 shows the same contrast for DeBERTa-v3-large (74.18 vs 35.35 on SciFact). This section reads the
saved scores of our students; it does not retrain them.

## 3. The testbed row: RankZephyr sliding windows on BM25 top-50 vs `ce`

`method.py --system rankzephyr` re-ranks the testbed's BM25 top-50 (ArguAna: top-20 after removing the query's own
document) with windows of 20 and step 10; `testbed.py score ... --against ce` gives the paired comparison below
(`results.json` in this folder). The 3,898 windows took about 13 hours on the laptop GPU; they are cached in `cache/` of
this folder, so `method.py` replays them in a few minutes without a GPU.

In [4]:
t = json.load(open(HERE / 'results.json', encoding='utf8'))
print(t['method'])
rows = ['| dataset | n | RankZephyr nDCG@10 | ce nDCG@10 | Δ | 95% CI | p | Recall@100 |', '|---|---|---|---|---|---|---|---|']
for ds, d in t['datasets'].items():
    v = d['ndcg10_vs_baseline']
    rows.append(f"| {ds} | {d['n_queries']} | {d['ndcg10']:.3f} | {d['baseline']['ndcg10']:.3f} | {v['delta']:+.3f} | [{v['ci95'][0]:+.3f}, {v['ci95'][1]:+.3f}] | {v['p']:.2g} | {d['recall100']:.3f} |")
display(Markdown('\n'.join(rows)))

RankGPT (RankZephyr-7B 4-bit; BM25 top-50, ArguAna top-20)


| dataset | n | RankZephyr nDCG@10 | ce nDCG@10 | Δ | 95% CI | p | Recall@100 |
|---|---|---|---|---|---|---|---|
| scifact | 300 | 0.740 | 0.688 | +0.052 | [+0.023, +0.081] | 0.0006 | 0.918 |
| nfcorpus | 323 | 0.361 | 0.347 | +0.014 | [+0.002, +0.026] | 0.026 | 0.246 |
| arguana | 1406 | 0.504 | 0.422 | +0.082 | [+0.062, +0.100] | 6.8e-17 | 0.969 |

Reading: RankZephyr beats `ce` on all three sets while reading 57–78% of the words `ce` reads (its passages
are cut to fit the 4,096-token context). Recall@100 hardly moves (at most 0.0001): both systems re-order the BM25 top-100. On
ArguAna, RankZephyr re-ranks only the top-20 while `ce` re-ranks the top-100, and `ce` loses more the deeper it re-ranks
there: against `ce` on the same top-20, the difference is +0.044 [+0.027, +0.061] (p = 5e-7, `testbed.compare`), about
half of the +0.082 above. RankZephyr itself stays about 1 point below BM25 on ArguAna (−0.012, p = 0.054); the target
there is a counter-argument, a kind of relevance absent from MS MARCO, on which RankZephyr was trained.

## 4. Cross-hardware check: the same runs on a Colab A100

Sections 1–3 ran on a laptop GPU (RTX 4060, 8 GB). We reran the whole reproduction (sections 1–3, the cross-encoder
controls and Table 4 with Flan-T5-large) in one Colab session on an A100, with the same code, data, prompts, models and
4-bit quantisation; the GPU and the operating system (Windows, Linux) differed. `results_colab.json` is the Colab testbed row, scored by the same
`testbed.py score --against ce`; `reproduction/results/replication_summary.json` compares all 32 runs query by query
(nDCG@10 ×100 in the second table).

In [5]:
lap = json.load(open(HERE / 'results.json', encoding='utf8'))
col = json.load(open(HERE / 'results_colab.json', encoding='utf8'))
rows = ['| dataset | laptop nDCG@10 | Colab nDCG@10 | laptop Δ vs ce [95% CI], p | Colab Δ vs ce [95% CI], p |', '|---|---|---|---|---|']
for ds in lap['datasets']:
    cells = []
    for t in (lap, col):
        v = t['datasets'][ds]['ndcg10_vs_baseline']
        cells.append(f"{v['delta']:+.3f} [{v['ci95'][0]:+.3f}, {v['ci95'][1]:+.3f}], p = {v['p']:.2g}")
    rows.append(f"| {ds} | {lap['datasets'][ds]['ndcg10']:.3f} | {col['datasets'][ds]['ndcg10']:.3f} | {cells[0]} | {cells[1]} |")
display(Markdown('\n'.join(rows)))

rep = json.load(open(REPRO / 'results' / 'replication_summary.json', encoding='utf8'))
runs = [r for r in rep['runs'] if r.get('colab') is not None]
print(f"replicated runs: {len(runs)} of {len(rep['runs'])}; identical query by query: {sum(r['same_queries'] == r['n'] for r in runs)}")
rows = ['| part | run | laptop | Colab | Colab − laptop [95% CI], p | same nDCG@10 | windows with the same answer |', '|---|---|---|---|---|---|---|']
for r in runs:
    diff = 'identical' if r['same_queries'] == r['n'] else f"{100*r['delta']:+.2f} [{100*r['ci95'][0]:+.2f}, {100*r['ci95'][1]:+.2f}], p = {r['p']:.2g}"
    w = r.get('windows')
    wins = f"{w['same_answer']} of {w['colab_windows']}" if w else ''
    rows.append(f"| {r['part']} | {r['run']} | {pct(r['laptop'])} | {pct(r['colab'])} | {diff} | {r['same_queries']} of {r['n']} | {wins} |")
display(Markdown('\n'.join(rows)))

| dataset | laptop nDCG@10 | Colab nDCG@10 | laptop Δ vs ce [95% CI], p | Colab Δ vs ce [95% CI], p |
|---|---|---|---|---|
| scifact | 0.740 | 0.739 | +0.052 [+0.023, +0.081], p = 0.0006 | +0.051 [+0.023, +0.080], p = 0.00057 |
| nfcorpus | 0.361 | 0.361 | +0.014 [+0.002, +0.026], p = 0.026 | +0.013 [+0.002, +0.025], p = 0.023 |
| arguana | 0.504 | 0.504 | +0.082 [+0.062, +0.100], p = 6.8e-17 | +0.082 [+0.062, +0.101], p = 5.3e-17 |

replicated runs: 32 of 32; identical query by query: 14


| part | run | laptop | Colab | Colab − laptop [95% CI], p | same nDCG@10 | windows with the same answer |
|---|---|---|---|---|---|---|
| RankZephyr testbed runs | scifact top-50 | 73.98 | 73.92 | -0.06 [-0.40, +0.28], p = 0.74 | 296 of 300 | 1086 of 1200 |
| RankZephyr testbed runs | nfcorpus top-50 | 36.09 | 36.06 | -0.03 [-0.33, +0.25], p = 0.83 | 312 of 323 | 1194 of 1292 |
| RankZephyr testbed runs | arguana top-20 | 50.40 | 50.42 | +0.03 [-0.05, +0.11], p = 0.5 | 1398 of 1406 | 1341 of 1406 |
| RankZephyr NovelEval, Table 3 | shuffled seed0 | 89.97 | 89.97 | identical | 21 of 21 | 17 of 21 |
| RankZephyr NovelEval, Table 3 | shuffled seed1 | 89.42 | 89.42 | identical | 21 of 21 | 20 of 21 |
| RankZephyr NovelEval, Table 3 | shuffled seed2 | 88.12 | 88.12 | identical | 21 of 21 | 21 of 21 |
| RankZephyr NovelEval, Table 3 | bm25 | 88.89 | 88.89 | identical | 21 of 21 | 21 of 21 |
| RankZephyr NovelEval, Table 3 | released | 89.46 | 89.46 | identical | 21 of 21 | 21 of 21 |
| cross-encoder controls | scifact top-50 | 68.53 | 68.53 | identical | 300 of 300 |  |
| cross-encoder controls | scifact top-100 | 68.81 | 68.81 | identical | 300 of 300 |  |
| cross-encoder controls | nfcorpus top-50 | 34.61 | 34.61 | +0.00 [+0.00, +0.00], p = 0.32 | 322 of 323 |  |
| cross-encoder controls | nfcorpus top-100 | 34.66 | 34.67 | +0.00 [+0.00, +0.00], p = 0.32 | 322 of 323 |  |
| cross-encoder controls | arguana top-20 | 45.98 | 45.99 | +0.01 [-0.06, +0.07], p = 0.85 | 1399 of 1406 |  |
| cross-encoder controls | arguana top-50 | 43.61 | 43.57 | -0.05 [-0.11, +0.00], p = 0.13 | 1401 of 1406 |  |
| cross-encoder controls | arguana top-100 | 42.23 | 42.21 | -0.03 [-0.07, +0.00], p = 0.23 | 1402 of 1406 |  |
| cross-encoder controls | noveleval | 77.08 | 77.08 | identical | 21 of 21 |  |
| distillation, Table 7: minilm_chatgpt | scifact | 64.48 | 65.20 | +0.73 [-0.18, +1.68], p = 0.13 | 251 of 300 |  |
| distillation, Table 7: minilm_chatgpt | nfcorpus | 33.62 | 33.27 | -0.36 [-0.78, +0.04], p = 0.084 | 190 of 323 |  |
| distillation, Table 7: minilm_chatgpt | arguana | 27.88 | 28.00 | +0.12 [-0.51, +0.74], p = 0.71 | 1020 of 1406 |  |
| distillation, Table 7: minilm_chatgpt | noveleval | 72.47 | 75.89 | +3.42 [+1.29, +5.90], p = 0.012 | 7 of 21 |  |
| distillation, Table 7: minilm_marco_binary | scifact | 21.62 | 16.86 | -4.77 [-6.88, -2.76], p = 7.2e-06 | 196 of 300 |  |
| distillation, Table 7: minilm_marco_binary | nfcorpus | 22.42 | 18.78 | -3.64 [-4.82, -2.49], p = 2.5e-09 | 135 of 323 |  |
| distillation, Table 7: minilm_marco_binary | arguana | 2.55 | 2.23 | -0.32 [-0.75, +0.11], p = 0.14 | 1294 of 1406 |  |
| distillation, Table 7: minilm_marco_binary | noveleval | 42.13 | 45.02 | +2.89 [-0.86, +6.21], p = 0.14 | 0 of 21 |  |
| Flan-T5 prompting, Table 4, SciFact 100 queries | pointwise.yes_no | 51.50 | 51.50 | identical | 100 of 100 |  |
| Flan-T5 prompting, Table 4, SciFact 100 queries | pointwise.qlm | 70.17 | 70.19 | +0.02 [+0.00, +0.05], p = 0.32 | 99 of 100 |  |
| Flan-T5 prompting, Table 4, SciFact 100 queries | listwise.likelihood | 66.79 | 66.98 | +0.19 [+0.00, +0.51], p = 0.17 | 98 of 100 |  |
| Flan-T5 prompting, Table 4, SciFact 100 queries | listwise.generation | 68.32 | 68.32 | identical | 100 of 100 |  |
| Flan-T5 prompting, Table 4, SciFact 100 queries | setwise.heapsort | 63.21 | 63.21 | identical | 100 of 100 |  |
| Flan-T5 prompting, Table 4, SciFact 100 queries | setwise.bubblesort | 70.22 | 70.22 | identical | 100 of 100 |  |
| Flan-T5 prompting, Table 4, SciFact 100 queries | pairwise.heapsort | 74.71 | 74.71 | identical | 100 of 100 |  |
| Flan-T5 prompting, Table 4, SciFact 100 queries | pairwise.bubblesort | 75.90 | 75.90 | identical | 100 of 100 |  |

Reading: RankZephyr's 4-bit generation is not bit-identical across the two GPUs. Given the same 20 prompts, 3 of
the 20 answers differed (`reproduction/results/generation_check_scifact_4bit_float16.json`), and over a full run
between 4.6% (ArguAna) and 9.5% (SciFact) of the windows gave a different permutation. The rankings barely move: every
testbed number stays within 0.001 nDCG@10 of the laptop run (p ≥ 0.5 for the difference), and the gains over `ce` keep
their size and significance. NovelEval and six of the eight Flan-T5 methods are identical, the two others within 0.2
points. Only the MiniLM students were retrained on Colab, and their training is not deterministic across GPUs: the
ChatGPT-distilled student moves by less than one point on the three testbed sets (3.4 points on the 21 NovelEval
questions), and the student trained on MS MARCO labels loses 3.6 and 4.8 points on NFCorpus and SciFact. The gap
between the two students, the point of Table 7, holds on both machines.